# Simple back scratch

1. **Kernel → Restart Kernel and Clear Outputs** first.
2. Run cell 1 and cell 2 (Shift+Enter). Cell 2 prints `Connected`.
3. Run **Scratch**. It goes to your scratch pose, wiggles left-right, opens and closes the gripper, and repeats. After `MINUTES` it goes to the sleep pose.

**■ Stop** in the toolbar = stop now and go to sleep pose. Arm power plug = emergency stop.

## 1. Settings

In [ ]:
import time

SIM = False          # True = test without moving the arm

# Where it scratches (your numbers from "Read angles")
SCRATCH_POSE = [180, 31, 149, 135, 95, 150]

# Where it goes to sleep after the time is up.
# !! Put your real sleep numbers here (fold the arm away, read them with "Read angles").
SLEEP_POSE = [180, 31, 149, 135, 95, 150]

MINUTES = 10         # how long it scratches, then sleeps

# Left-right wiggle
WIGGLE_SERVO = 1     # 1 = base. Try 5 (wrist turn) or 4 (wrist bend) if you prefer
WIGGLE_DEG = 10      # how far it moves ("a little"). Smaller = softer
WIGGLE_MS = 400      # time per move. Bigger = slower, gentler
WIGGLES = 3          # left-right moves before opening the gripper

# Gripper
GRIP_CLOSED = 150
GRIP_OPEN = 90
GRIP_MS = 400


## 2. Program + connect (just run it)

In [ ]:
def clamp(sid, a):
    hi = 270 if sid == 5 else 180
    return int(max(0, min(hi, a)))

class SimArm:
    def Arm_serial_servo_write6_array(self, a, t): print(f"  [sim] all -> {a} ({t} ms)")
    def Arm_serial_servo_write(self, s, a, t): print(f"  [sim] servo {s} -> {a} ({t} ms)")

def connect():
    if SIM:
        return SimArm()
    from Arm_Lib import Arm_Device
    a = Arm_Device()
    time.sleep(0.1)
    return a

def wait(ms):
    time.sleep(0 if SIM else ms / 1000 + 0.05)

def go(pose, ms):
    pose = [clamp(i + 1, v) for i, v in enumerate(pose)]
    arm.Arm_serial_servo_write6_array(pose, ms)
    wait(ms)

def servo(sid, angle, ms):
    arm.Arm_serial_servo_write(sid, clamp(sid, angle), ms)
    wait(ms)

def sleep_pose():
    print("Going to sleep pose...")
    go(SLEEP_POSE, 2000)

def scratch():
    base = SCRATCH_POSE[WIGGLE_SERVO - 1]
    # wiggle around the start angle; if it's at the end of the range (like 180),
    # move only to the side that is possible
    hi = 270 if WIGGLE_SERVO == 5 else 180
    a = clamp(WIGGLE_SERVO, base - WIGGLE_DEG)
    b = clamp(WIGGLE_SERVO, base + WIGGLE_DEG)
    if b == base:      # can't go further up -> wiggle down and back
        b = base
    if a == base:
        a = base
    print(f"Scratching for {MINUTES} min (servo {WIGGLE_SERVO}: {a} <-> {b}). Press Stop to end.")
    go(SCRATCH_POSE, 2000)
    end = time.time() + MINUTES * 60
    rounds = 0
    try:
        while time.time() < end:
            for _ in range(WIGGLES):
                servo(WIGGLE_SERVO, a, WIGGLE_MS)
                servo(WIGGLE_SERVO, b, WIGGLE_MS)
            servo(WIGGLE_SERVO, base, WIGGLE_MS)
            servo(6, GRIP_OPEN, GRIP_MS)
            servo(6, GRIP_CLOSED, GRIP_MS)
            rounds += 1
            if SIM and rounds >= 2:
                break
        print("Time's up.")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        sleep_pose()

arm = connect()
print("Connected" if not SIM else "Simulation mode")


## Scratch

In [ ]:
scratch()

## Sleep now

In [ ]:
sleep_pose()